# PPO rollout — CPU env-workers + single GPU agent (inference server)

`N` CPU processes each drive their own orbit_wars game: env.step + the (now
vectorized) featurizer, then send the raw T=10 window to **one GPU process that
holds the single agent instance**. The GPU process dynamically batches the
pending (worker, seat) requests, runs ONE forward (L0 + L1–L4 + PairHead), and
scatters pair_logits/pair_frac back. Workers hold **no model**. Forward goes where
batching pays (GPU); env+featurize go where parallelism pays (CPU cores).

Validated locally on CPU (architecture/IPC/parity). This runs it on the Colab GPU.

## 1. Setup — auth, pull code + ckpts + the server, check GPU

In [ ]:
from google.colab import auth
auth.authenticate_user()
import os, subprocess
PROJECT = 'analog-receiver-489214-e9'
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT], check=True)
os.makedirs('/content/ow', exist_ok=True)
os.chdir('/content/ow')

In [ ]:
# code (smoke/featurizer incl. the vectorized ETA), ckpts, and the server
!gcloud storage cp gs://orbit-wars-shipping/entity/code.tgz . && tar xzf code.tgz
!gcloud storage cp gs://orbit-wars-shipping/tmp/bench_ckpts.tgz . && tar xzf bench_ckpts.tgz   # -> ckpts/{planet,fleet,comet,entity_encoder_best.pt}
!gcloud storage cp gs://orbit-wars-shipping/tmp/inference_server.py .
!pip -q install 'kaggle_environments==1.28.1'

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
import os; print('cpus', os.cpu_count())

## 2. Config

- **WORKERS**: CPU env-workers. Leave 1–2 cores for the GPU server + main (Colab Pro ≈ 8 vCPU).
- **GAMES**: total games (alternates 2p/4p across `SETTINGS`).
- **MAX_BATCH / BATCH_WINDOW_MS**: GPU dynamic-batching — the server waits up to the
  window to gather more requests before forwarding. Bigger window → bigger batches
  (better GPU use) but more per-step latency. With 1 env/worker the batch ≈ workers×seats.
- Forward is on the GPU, so featurize stops competing with it for CPU memory bandwidth
  → it should run near its single-core ~10 ms.

In [ ]:
WORKERS         = max(2, (os.cpu_count() or 4) - 1)
GAMES           = 16
SETTINGS        = '2,4'      # alternated across games
MAX_PLANETS     = 32
MAX_FLEETS      = 256
WINDOW          = 10         # T=10 history
MAX_BATCH       = 64
BATCH_WINDOW_MS = 5.0
print(f'workers={WORKERS} games={GAMES} settings={SETTINGS} max_batch={MAX_BATCH}')

## 3. Run the inference-server rollout

Prints throughput (env-steps/s, games/min), GPU forward batch stats (mean/max batch,
fwd ms), and the per-step CPU breakdown (featurize / wait-for-GPU / decode / env).

In [ ]:
import os
os.environ['OW_REPO'] = '/content/ow'
cmd = (f"cd /content/ow && OW_REPO=/content/ow python inference_server.py "
       f"--device cuda --workers {WORKERS} --games {GAMES} --settings {SETTINGS} "
       f"--ckpt /content/ow/ckpts/entity_encoder_best.pt --ck-dir /content/ow/ckpts "
       f"--max-planets {MAX_PLANETS} --max-fleets {MAX_FLEETS} --window {WINDOW} "
       f"--max-batch {MAX_BATCH} --batch-window-ms {BATCH_WINDOW_MS} --gpu-threads 2")
print(cmd)
get_ipython().system(cmd)

## Notes / tuning

- **Throughput is CPU-worker-bound** (the GPU forward is amortized), so it scales with
  `WORKERS` until the cores or the GPU saturate. On Colab (≈8 vCPU) you're core-limited;
  a many-vCPU GPU VM (e.g. `n1-standard-32 + T4`) feeds the GPU far better.
- **Small batches?** With 1 env/worker the batch ≈ workers×seats and desync keeps it low.
  Raise `BATCH_WINDOW_MS`, add workers, or extend the worker to run several envs each
  (more outstanding requests per worker → bigger batches, better GPU utilization).
- **fork + CUDA**: CUDA is initialized only inside the GPU-server child (after fork); the
  main process and workers never touch CUDA. If your Colab image dislikes fork+CUDA,
  switch the context to `spawn` in `run_rollout` (all worker/server fns are top-level).
- Producing shards: the workers currently return per-game stats. To emit PPO shards,
  collect the learner-seat `StepRecord`s into an `EpisodeBuffer` and `save_shard` (shards.py).